# CORE-ave: Evaluation & Analysis

Comprehensive evaluation including:
- Overall metrics (Recall@20, MRR@20)
- Grouped metrics by prefix length
- Grouped metrics by target item popularity
- Comparison with baselines
- Qualitative analysis of predictions
- Efficiency metrics

In [ ]:
import json
import torch
import numpy as np
import pandas as pd
from collections import Counter
from pathlib import Path
import sys
sys.path.append("../../src")

from models.core_ave import CoreAve, create_core_ave
from data.datasets import create_dataloaders, SessionDataset
from evaluation.metrics import (
    evaluate_model, recall_at_k, mrr_at_k,
    group_by_prefix_length, group_by_target_popularity,
    PopularityRecommender, TransitionRecommender, SessionPopularityRecommender
)

In [ ]:
# Load Data & Vocabulary
train_df = pd.read_parquet("../../data/processed/train.parquet")
val_df = pd.read_parquet("../../data/processed/validation.parquet")
test_df = pd.read_parquet("../../data/processed/test.parquet")

with open("../../data/processed/item_mapping.json") as f:
    mapping = json.load(f)
item2idx = mapping['item2idx']
idx2item = {int(k): v for k, v in mapping['idx2item'].items()}
vocab_size = mapping['vocab_size']

print(f"Train: {len(train_df):,} pairs")
print(f"Val: {len(val_df):,} pairs")
print(f"Test: {len(test_df):,} pairs")
print(f"Vocab size: {vocab_size}")

In [ ]:
# Build Train Frequency for Popularity Grouping
train_items = []
for row in train_df.itertuples():
    train_items.extend(row.prefix_items)
    train_items.append(row.target_item)
train_freq = Counter(train_items)

# Build Transitions
transitions = {}
for row in train_df.itertuples():
    if len(row.prefix_items) > 0:
        last = row.prefix_items[-1]
        nxt = row.target_item
        if last not in transitions:
            transitions[last] = Counter()
        transitions[last][nxt] += 1

print(f"Train vocab size: {len(train_freq)}")
print(f"Transition pairs: {len(transitions)}")

In [ ]:
# Load Trained Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = create_core_ave(vocab_size=vocab_size).to(device)
checkpoint = torch.load("../../checkpoints/core_ave/core_ave_seed42_best.pt", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(f"Loaded model from epoch {checkpoint['epoch']}")
print(f"Best val metric: {checkpoint['best_metric']:.4f}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Evaluate CORE-ave on Test Set
print("Evaluating CORE-ave on test set...")
core_results = evaluate_model(
    model, test_df, k=20,
    item2idx=item2idx, idx2item=idx2item,
    verbose=True
)

# Add popularity grouping
core_results['by_target_popularity'] = group_by_target_popularity(
    core_results['per_example'], train_freq
)

print(f"\nCORE-ave Results:")
print(f"  Recall@20: {core_results['overall']['recall@20']:.4f}")
print(f"  MRR@20: {core_results['overall']['mrr@20']:.4f}")

In [ ]:
# Evaluate Baselines
print("\nEvaluating baselines...")

pop_model = PopularityRecommender(train_freq, k=20)
pop_results = evaluate_model(pop_model, test_df, k=20, verbose=True)
pop_results['by_target_popularity'] = group_by_target_popularity(pop_results['per_example'], train_freq)

trans_model = TransitionRecommender(transitions, k=20)
trans_results = evaluate_model(trans_model, test_df, k=20, verbose=True)
trans_results['by_target_popularity'] = group_by_target_popularity(trans_results['per_example'], train_freq)

sesspop_model = SessionPopularityRecommender(train_freq, k=20)
sesspop_results = evaluate_model(sesspop_model, test_df, k=20, verbose=True)
sesspop_results['by_target_popularity'] = group_by_target_popularity(sesspop_results['per_example'], train_freq)

In [ ]:
# Overall Comparison Table
print("\n" + "="*70)
print("OVERALL METRICS COMPARISON")
print("="*70)
print(f"{'Model':<20} | {'Recall@20':>10} | {'MRR@20':>10}")
print("-"*70)

models = [
    ("CORE-ave", core_results),
    ("Transition", trans_results),
    ("Session-POP", sesspop_results),
    ("POP", pop_results),
]

for name, res in models:
    print(f"{name:<20} | {res['overall']['recall@20']:>10.4f} | {res['overall']['mrr@20']:>10.4f}")

In [ ]:
# By Prefix Length Comparison
print("\n" + "="*70)
print("BY PREFIX LENGTH")
print("="*70)

bins = ['1', '2-3', '4-6', '7+']

print(f"{'Model':<15} | {'1':^15} | {'2-3':^15} | {'4-6':^15} | {'7+':^15}")
print("-"*80)

for name, res in models:
    row = f"{name:<15}"
    for b in bins:
        m = res['by_prefix_length'].get(b, {'recall@20': 0, 'mrr@20': 0, 'count': 0})
        row += f" | R={m['recall@20']:.3f} M={m['mrr@20']:.3f} n={m['count']}"
    print(row)

In [ ]:
# By Target Popularity Comparison
print("\n" + "="*70)
print("BY TARGET POPULARITY (Train Frequency Tertiles)")
print("="*70)

pop_bins = ['head', 'torso', 'tail']

print(f"{'Model':<15} | {'Head':^20} | {'Torso':^20} | {'Tail':^20}")
print("-"*80)

for name, res in models:
    row = f"{name:<15}"
    for b in pop_bins:
        m = res['by_target_popularity'].get(b, {'recall@20': 0, 'mrr@20': 0, 'count': 0, 'freq_range': (0,0)})
        row += f" | R={m['recall@20']:.3f} M={m['mrr@20']:.3f} n={m['count']}"
    print(row)

In [ ]:
# Efficiency Metrics
print("\n" + "="*70)
print("EFFICIENCY METRICS")
print("="*70)

import time

# Training time per epoch (from training logs)
print("Training (batch=2048, GPU):")
print(f"  CORE-ave: ~75s/epoch")
print(f"  CORE-trm: ~132s/epoch")

# Inference latency
model.eval()
test_loader = create_dataloaders(
    train_path="../../data/processed/train.parquet",
    val_path="../../data/processed/validation.parquet",
    test_path="../../data/processed/test.parquet",
    item_mapping_path="../../data/processed/item_mapping.json",
    batch_size=1024,
    max_len=50,
    num_workers=0
)[2]

latencies = []
with torch.no_grad():
    for prefix, target, plen in test_loader:
        prefix = prefix.to(device)
        plen = plen.to(device)

        start = time.time()
        _ = model.recommend(prefix, plen, k=20)
        latencies.append((time.time() - start) * 1000 / prefix.size(0))

        if len(latencies) >= 100:
            break

print(f"\nInference latency (per sample, batch=1024):")
print(f"  Mean: {np.mean(latencies):.2f} ms")
print(f"  Std:  {np.std(latencies):.2f} ms")
print(f"  P95:  {np.percentile(latencies, 95):.2f} ms")

In [ ]:
# Qualitative Analysis: Inspect Predictions
print("\n" + "="*70)
print("QUALITATIVE ANALYSIS")
print("="*70)

def get_item_name(item_id):
    # Check if in our catalog
    catalog = {
        81766: "Wireless Headphones Pro",
        31331: "Bluetooth Speaker Mini",
        32118: "USB-C Hub 7-in-1",
        9654: "Mechanical Keyboard RGB",
        5103: "Ergonomic Mouse Vertical",
        69597: "27\" 4K Monitor IPS",
        4168: "Laptop Stand Aluminum",
        79898: "Webcam 1080p Streaming",
        36083: "External SSD 1TB NVMe",
        10858: "WiFi 6 Router AX3000",
        14614: "Smart Desk Lamp LED",
        6848: "Cable Management Kit",
        79141: "Monitor Arm Dual",
        12345: "Gaming Chair Ergonomic",
        34567: "Keycap Set PBT Cherry",
        78901: "RGB Light Strip 2m",
        89012: "Microphone Arm Boom",
        1234: "Acoustic Panels 12pk",
        5678: "Stream Deck Mini",
        9012: "Green Screen Collapsible",
    }
    return catalog.get(item_id, f"Item {item_id}")

# Sample correct predictions
correct = [ex for ex in core_results['per_example'] if ex['recall@20'] == 1.0]
incorrect = [ex for ex in core_results['per_example'] if ex['recall@20'] == 0.0]

print(f"Correct predictions: {len(correct)} / {len(core_results['per_example'])}")
print(f"Incorrect predictions: {len(incorrect)} / {len(core_results['per_example'])}")

print("\n--- Sample CORRECT predictions ---")
for ex in correct[:5]:
    session = [get_item_name(x) for x in ex['prefix_items'][-3:]]
    target = get_item_name(ex['target_item'])
    recs = [get_item_name(x) for x in ex['recommended'][:5]]
    rank = ex['recommended'].index(ex['target_item']) + 1
    print(f"  Session: {' -> '.join(session)} -> Target: {target}")
    print(f"  Top-5: {recs} (Target rank: {rank})")
    print()

In [ ]:
print("--- Sample INCORRECT predictions ---")
for ex in incorrect[:5]:
    session = [get_item_name(x) for x in ex['prefix_items'][-3:]]
    target = get_item_name(ex['target_item'])
    recs = [get_item_name(x) for x in ex['recommended'][:5]]
    print(f"  Session: {' -> '.join(session)} -> Target: {target}")
    print(f"  Top-5: {recs}")
    print()

In [ ]:
# Analysis by Session Patterns
print("\n" + "="*70)
print("ANALYSIS BY SESSION PATTERNS")
print("="*70)

# Repeated items in session
repeated_correct = 0
repeated_total = 0
single_correct = 0
single_total = 0
long_correct = 0
long_total = 0

for ex in core_results['per_example']:
    prefix = ex['prefix_items']
    is_correct = ex['recall@20'] == 1.0

    # Check for repeated items
    has_repeats = len(prefix) != len(set(prefix))
    is_long = len(prefix) >= 7
    is_single = len(prefix) == 1

    if has_repeats:
        repeated_total += 1
        if is_correct:
            repeated_correct += 1

    if is_single:
        single_total += 1
        if is_correct:
            single_correct += 1

    if is_long:
        long_total += 1
        if is_correct:
            long_correct += 1

print(f"Sessions with repeated items: {repeated_correct}/{repeated_total} = {repeated_correct/repeated_total*100:.1f}%")
print(f"Single-item sessions: {single_correct}/{single_total} = {single_correct/single_total*100:.1f}%")
print(f"Long sessions (>=7): {long_correct}/{long_total} = {long_correct/long_total*100:.1f}%" if long_total > 0 else "Long sessions: 0")

In [ ]:
# Save Evaluation Results
eval_output = {
    "model": "core_ave",
    "overall": core_results["overall"],
    "by_prefix_length": core_results["by_prefix_length"],
    "by_target_popularity": core_results["by_target_popularity"],
    "baselines": {
        "pop": {
            "overall": pop_results["overall"],
            "by_prefix_length": pop_results["by_prefix_length"],
            "by_target_popularity": pop_results["by_target_popularity"]
        },
        "transition": {
            "overall": trans_results["overall"],
            "by_prefix_length": trans_results["by_prefix_length"],
            "by_target_popularity": trans_results["by_target_popularity"]
        },
        "session_pop": {
            "overall": sesspop_results["overall"],
            "by_prefix_length": sesspop_results["by_prefix_length"],
            "by_target_popularity": sesspop_results["by_target_popularity"]
        }
    },
    "qualitative": {
        "correct_predictions": len(correct),
        "incorrect_predictions": len(incorrect),
        "repeated_items_accuracy": repeated_correct / repeated_total if repeated_total > 0 else 0,
        "single_item_accuracy": single_correct / single_total if single_total > 0 else 0,
        "long_session_accuracy": long_correct / long_total if long_total > 0 else 0
    }
}

with open("../../reports/models/core_ave_evaluation.json", 'w') as f:
    json.dump(eval_output, f, indent=2, default=str)

print("\nEvaluation results saved to ../../reports/models/core_ave_evaluation.json")